# dim_fecha
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Gold (nivel 2) · dimensión Fecha |
| Propósito | Calendario diario 2024-2027 para analizar por año, trimestre, mes y día. Clave legible AAAAMMDD (sk_fecha). |
| Fuente | generada (no depende de silver) |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.0 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import functions as F

from comun_gold import sk_fecha, spark

## 2. Tabla `dim_fecha`
Calendario diario (2024-2027) para analizar por año, mes y día.

In [ ]:
@dlt.table(name="dim_fecha", comment="Calendario diario (2024-2027) para analizar por año, mes y día.")
def dim_fecha():
    dias = spark.sql("SELECT explode(sequence(DATE'2024-01-01', DATE'2027-12-31', INTERVAL 1 DAY)) AS fecha")
    return dias.select(
        sk_fecha("fecha").alias("sk_fecha"), "fecha",
        F.year("fecha").alias("anio"), F.quarter("fecha").alias("trimestre"),
        F.month("fecha").alias("mes"), F.date_format("fecha", "MMMM").alias("nombre_mes"),
        F.dayofmonth("fecha").alias("dia"), F.dayofweek("fecha").alias("dia_semana"),
        F.dayofweek("fecha").isin(1, 7).alias("es_fin_de_semana"))